# Plugin 顺序语义场景手册

这个 notebook 演示 `Solver` 控制面的顺序规则与运行时护栏。


In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'nsgablack').exists() and (ROOT.parent / 'nsgablack').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from nsgablack.core.base import BlackBoxProblem
from nsgablack.core.blank_solver import SolverBase
from nsgablack.plugins.base import Plugin


In [ ]:
class DemoProblem(BlackBoxProblem):
    def __init__(self):
        super().__init__(dimension=2, bounds=[(-1.0, 1.0), (-1.0, 1.0)])

    def evaluate(self, x):
        arr = np.asarray(x, dtype=float)
        return np.asarray([float(np.sum(arr * arr))], dtype=float)

class NamedPlugin(Plugin):
    def __init__(self, name: str, priority: int = 0):
        super().__init__(name=name, priority=priority)

def plugin_names(solver: SolverBase):
    return [str(p.name) for p in solver.plugin_manager.plugins]


## 场景 1：priority 小值先执行


In [ ]:
s = SolverBase(problem=DemoProblem())
s.add_plugin(NamedPlugin('p10', priority=10))
s.add_plugin(NamedPlugin('p0', priority=0))
s.add_plugin(NamedPlugin('p5', priority=5))
plugin_names(s)


## 场景 2：未知组件名（即时报错）


In [ ]:
s = SolverBase(problem=DemoProblem())
s.add_plugin(NamedPlugin('a', priority=0))
try:
    s.set_plugin_order('a', after=('missing_component',))
except Exception as e:
    print(type(e).__name__)
    print(e)


## 场景 3：priority 方向冲突（严格拒绝）


In [ ]:
s = SolverBase(problem=DemoProblem())
s.add_plugin(NamedPlugin('high', priority=0))
s.add_plugin(NamedPlugin('low', priority=10))
try:
    s.set_plugin_order('high', after=('low',))
except Exception as e:
    print(type(e).__name__)
    print(e)


## 场景 4：运行中直接改拓扑（报错）


In [ ]:
s = SolverBase(problem=DemoProblem())
s.add_plugin(NamedPlugin('a', priority=0))
s.add_plugin(NamedPlugin('b', priority=0))
s.running = True
try:
    s.set_plugin_order('a', after=('b',))
except Exception as e:
    print(type(e).__name__)
    print(e)
finally:
    s.running = False


## 场景 5：request_plugin_order 在边界生效


In [ ]:
class RequestOrderPlugin(Plugin):
    def __init__(self):
        super().__init__(name='request', priority=100)

    def on_generation_end(self, generation: int):
        if generation == 0 and self.solver is not None:
            self.solver.request_plugin_order('a', after=('b',))

s = SolverBase(problem=DemoProblem())
s.add_plugin(NamedPlugin('a', priority=0))
s.add_plugin(NamedPlugin('b', priority=0))
s.add_plugin(RequestOrderPlugin())
print('run 前:', plugin_names(s))
s.run(max_steps=2)
print('run 后:', plugin_names(s))


## 场景 6：嵌套 solver 的作用域隔离


In [ ]:
parent = SolverBase(problem=DemoProblem())
child = SolverBase(problem=DemoProblem())

parent.add_plugin(NamedPlugin('parent_archive', priority=0))
parent.add_plugin(NamedPlugin('parent_report', priority=1), after=('parent_archive',))

child.add_plugin(NamedPlugin('child_eval', priority=0))
child.add_plugin(NamedPlugin('child_export', priority=1), after=('child_eval',))

print('parent order:', plugin_names(parent))
print('child order: ', plugin_names(child))

print('cross-scope reference on parent (expected fail):')
try:
    parent.set_plugin_order('parent_report', after=('child_eval',))
except Exception as e:
    print(type(e).__name__)
    print(e)


### 结论

- 父层和子层各自维护自己的组件 DAG。
- 跨作用域组件名不会被解析，直接报错。
- 组件编排建议只在本 solver 作用域内完成。
